In [1]:
%pip install --upgrade --quit  langchain langchain-community langchain-openai lanchain-experimental

Note: you may need to restart the kernel to use updated packages.



Usage:   
  c:\Users\Shivam\Desktop\graphrag\graphrag\Scripts\python.exe -m pip install [options] <requirement specifier> [package-index-options] ...
  c:\Users\Shivam\Desktop\graphrag\graphrag\Scripts\python.exe -m pip install [options] -r <requirements file> [package-index-options] ...
  c:\Users\Shivam\Desktop\graphrag\graphrag\Scripts\python.exe -m pip install [options] [-e] <vcs project url> ...
  c:\Users\Shivam\Desktop\graphrag\graphrag\Scripts\python.exe -m pip install [options] [-e] <local project path> ...
  c:\Users\Shivam\Desktop\graphrag\graphrag\Scripts\python.exe -m pip install [options] <archive url/path> ...

no such option: --quit


In [24]:
from dotenv import load_dotenv
import os

load_dotenv()

print("URI:", os.getenv("NEO4J_URI"))
print("USERNAME:", os.getenv("NEO4J_USERNAME"))
print("DATABASE:", os.getenv("NEO4J_DATABASE"))
print("PASSWORD LOADED:", bool(os.getenv("NEO4J_PASSWORD")))

URI: neo4j+s://24717d57.databases.neo4j.io
USERNAME: 24717d57
DATABASE: 24717d57
PASSWORD LOADED: True


In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

OPEN_API_KEY = os.getenv("OPENAI_API_KEY")
NEO4J_URI=os.getenv("NEO4J_URI")
NEO4J_USERNAME=os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD=os.getenv("NEO4J_PASSWORD")


In [3]:
import truststore
truststore.inject_into_ssl()

from langchain_neo4j import Neo4jGraph  # updated package, per our earlier fix

graph = Neo4jGraph()


In [4]:
import requests

title = "Elizabeth_I"

url = f"https://en.wikipedia.org/w/rest.php/v1/page/{title}"

headers = {
    "User-Agent": "GraphRAG-Project/1.0"
}

response = requests.get(url, headers=headers)

print("Status:", response.status_code)

data = response.json()

raw_document = data["source"]

print(raw_document[:2000])
print("Length:", len(raw_document))

Status: 200
{{Short description|Queen of England and Ireland from 1558 to 1603}}
{{Redirect-multi|2|Elizabeth of England|Elizabeth Tudor||Elizabeth I (disambiguation)|and|Elizabeth of England (disambiguation)|and|Elizabeth Tudor (disambiguation)}}
{{featured article}}
{{Pp-vandalism|small=yes}}
{{Pp-move}}
{{Use British English|date=September 2013}}
{{Use dmy dates|date=July 2026}}
{{Infobox royalty
| name         = Elizabeth I
| image        = Darnley stage 3.jpg
| caption      = The ''[[Darnley Portrait]]'', {{Circa|1575}}
| alt          = Full-length portrait of Queen Elizabeth in her early 40s. She has red hair, fair skin, and wears a crown and a pearl necklace.
| succession   = [[List of English monarchs|Queen of England]] and [[List of Irish monarchs|Ireland]]
| more_text    = ([[Style of the English sovereigns|more...]])
| reign        = 17 November 1558{{sndash}}<br/>24 March 1603
| coronation   = 15 January 1559
| cor_type     = [[Coronation of Elizabeth I|Coronation]]
| prede

In [5]:
raw_document

'{{Short description|Queen of England and Ireland from 1558 to 1603}}\n{{Redirect-multi|2|Elizabeth of England|Elizabeth Tudor||Elizabeth I (disambiguation)|and|Elizabeth of England (disambiguation)|and|Elizabeth Tudor (disambiguation)}}\n{{featured article}}\n{{Pp-vandalism|small=yes}}\n{{Pp-move}}\n{{Use British English|date=September 2013}}\n{{Use dmy dates|date=July 2026}}\n{{Infobox royalty\n| name         = Elizabeth I\n| image        = Darnley stage 3.jpg\n| caption      = The \'\'[[Darnley Portrait]]\'\', {{Circa|1575}}\n| alt          = Full-length portrait of Queen Elizabeth in her early 40s. She has red hair, fair skin, and wears a crown and a pearl necklace.\n| succession   = [[List of English monarchs|Queen of England]] and [[List of Irish monarchs|Ireland]]\n| more_text    = ([[Style of the English sovereigns|more...]])\n| reign        = 17 November 1558{{sndash}}<br/>24 March 1603\n| coronation   = 15 January 1559\n| cor_type     = [[Coronation of Elizabeth I|Coronation]

In [11]:
len(raw_document)

132174

In [10]:
from langchain_text_splitters import TokenTextSplitter

text_splitter = TokenTextSplitter(chunk_size=512, chunk_overlap=24)

# if you just want text chunks back:
chunks = text_splitter.split_text(raw_document[0])  # split_text takes one string at a time

# if you want Document objects (recommended if you're feeding this into Neo4jGraph/embeddings next):
documents = text_splitter.create_documents(raw_document[:3])